In [3]:
import pandas

train_df = pandas.read_csv("../../data/processed/step2_lineage.csv", low_memory=False)
area_df  = pandas.read_csv("../../data/raw/hanwoo_area.csv")

print(f"train_df: {train_df.shape}")
print(f"area_df: {area_df.shape}")

# area_df -99 처리
area_cols = ["C2023", "C2024", "C2025", "AREA"]
for col in area_cols:
    area_df[col] = area_df[col].replace(-99, None)

# FARM_UNIQUE_NO 기준 합산 (복수 허가 건 합산)
area_summary = area_df.groupby("FARM_UNIQUE_NO").agg(
    C2023=("C2023", "sum"),
    C2024=("C2024", "sum"),
    C2025=("C2025", "sum"),
    AREA=("AREA", "sum")
).reset_index()

print(f"\narea_summary 행 수: {len(area_summary):,}")
print(f"area_summary FARM_UNIQUE_NO 중복: {area_summary['FARM_UNIQUE_NO'].duplicated().sum():,}")

before_rows = len(train_df)
before_cols = len(train_df.columns)

train_df = train_df.merge(
    area_summary,
    on="FARM_UNIQUE_NO",
    how="left"
)

after_rows = len(train_df)
after_cols = len(train_df.columns)

print(f"\n=== 병합 결과 ===")
print(f"병합 전: {before_rows:,}행 {before_cols}열")
print(f"병합 후: {after_rows:,}행 {after_cols}열")
print(f"\nAREA 결측: {train_df['AREA'].isnull().sum():,}")
print(f"C2023 결측: {train_df['C2023'].isnull().sum():,}")
print(f"C2024 결측: {train_df['C2024'].isnull().sum():,}")
print(f"C2025 결측: {train_df['C2025'].isnull().sum():,}")

# 저장
train_df.to_csv("../data/processed/step3_area.csv", index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {train_df.shape}")

train_df: (2408699, 31)
area_df: (91896, 5)

area_summary 행 수: 86,703
area_summary FARM_UNIQUE_NO 중복: 0

=== 병합 결과 ===
병합 전: 2,408,699행 31열
병합 후: 2,408,699행 35열

AREA 결측: 37,384
C2023 결측: 37,384
C2024 결측: 37,384
C2025 결측: 37,384

저장 완료: (2408699, 35)
